# PyCRMKit — Local Experimentation

Notebook de prise en main et de **smoke test local** de la version actuellement présente dans le dépôt.

Le dépôt est actuellement en **`0.3.0b2 — Lead Conversion`**. Ce notebook couvre uniquement les APIs réellement disponibles à ce jalon :

- CRM in-memory ;
- Contacts et Organisations ;
- Relationships ;
- Activities, Tasks et Timeline ;
- Leads (création / qualification / disqualification / conversion) ;
- Pipelines / Stages ;
- Opportunities et transitions de stage ;
- retry idempotent de la conversion Lead → Opportunity.

Les exemples suivent les contrats publics déjà utilisés par la documentation et les smoke/E2E tests du dépôt.

## 1. Préparer l'import local

Le notebook peut être lancé depuis la racine du dépôt ou depuis `notebooks/`. La cellule ci-dessous localise automatiquement la racine et ajoute `src/` au `sys.path`.

Pour un environnement de développement complet, l'installation recommandée reste :

```bash
python -m venv .venv
source .venv/bin/activate  # Windows: .venv\Scripts\activate
python -m pip install -e ".[dev]"
```


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pycrmkit").exists():
            return candidate
    raise RuntimeError("Racine du dépôt PyCRMKit introuvable.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repo root : {REPO_ROOT}")
print(f"Source    : {SRC_DIR}")


## 2. Vérifier la version chargée


In [ ]:
from pycrmkit import CRM, __version__

print("PyCRMKit version:", __version__)
if __version__ != "0.3.0b2":
    print("ℹ️ Ce notebook a été écrit pour 0.3.0b2 ; certaines cellules peuvent évoluer avec le framework.")

## 3. Créer un CRM in-memory

`CRM.memory()` est le mode idéal pour expérimenter : aucun serveur, aucune base de données et aucune dépendance framework ne sont nécessaires.


In [ ]:
crm = CRM.memory().with_context(
    actor_id="local-notebook",
    correlation_id="experiment-001",
)

print(type(crm).__name__)
print("actor_id      :", crm.context.actor_id)
print("correlation_id:", crm.context.correlation_id)


## 4. Contact et Organisation


In [ ]:
from pycrmkit.contacts import ContactEmail, ContactPhone

contact = crm.contacts.create(
    first_name="Ada",
    last_name="Lovelace",
    source="local-notebook",
    emails=(ContactEmail("Ada.Lovelace@Example.com", is_primary=True),),
    phones=(ContactPhone("+33 6 12 34 56 78", is_primary=True),),
)

organization = crm.organizations.create(
    legal_name="Analytical Engines SAS",
    trading_name="Analytical Engines",
    source="local-notebook",
)

print("Contact     :", contact.id, contact.display_name)
print("Email norm. :", contact.emails[0].normalized)
print("Phone norm. :", contact.phones[0].normalized)
print("Organization:", organization.id, organization.display_name)


## 5. Créer une relation Contact ↔ Organisation


In [ ]:
from pycrmkit.relationships import RelationshipEndpoint, RelationshipType

relationship = crm.relationships.create(
    source=RelationshipEndpoint.contact(contact.id),
    target=RelationshipEndpoint.organization(organization.id),
    relationship_type=RelationshipType("employee_of"),
    role="CTO",
    is_primary=True,
)

print("Relationship:", relationship.id)
print("Type        :", relationship.relationship_type)
print("Role        :", relationship.role)


## 6. Activity + Task + Timeline

Les `Activity` et `Task` restent des agrégats distincts. La Timeline est un **read model projeté à partir des événements métier**.


In [ ]:
from pycrmkit.activities import ActivityParticipant
from pycrmkit.core.references import EntityReference

contact_ref = EntityReference("contact", contact.id)
organization_ref = EntityReference("organization", organization.id)

activity = crm.activities.log(
    type="meeting",
    subject="Discovery meeting",
    direction="outbound",
    duration_seconds=45 * 60,
    participants=(
        ActivityParticipant(contact_ref, role="customer", is_primary=True),
    ),
    references=(organization_ref,),
    source="local-notebook",
)

task = crm.tasks.create(
    title="Envoyer la proposition",
    priority="high",
    references=(contact_ref, organization_ref),
    source="local-notebook",
)
completed_task = crm.tasks.complete(task.id)

print("Activity:", activity.id, activity.type.value, activity.subject)
print("Task    :", completed_task.id, completed_task.status.value)


In [ ]:
history = crm.timeline.for_contact(contact.id)

print(f"Timeline entries: {history.total}")
for entry in history.items:
    print(f"- {entry.occurred_at.isoformat()} | {entry.event_type} | {entry.title}")


## 7. Lead — capture et qualification

À `0.3.0b2`, le facade expose `create`, `qualify`, `disqualify` et désormais `convert`. La conversion sera testée après la définition du Pipeline.

In [ ]:
lead = crm.leads.create(
    contact_id=contact.id,
    organization_id=organization.id,
    source="website",
)
qualified_lead = crm.leads.qualify(lead.id)

print("Lead:", qualified_lead.id)
print("Status:", qualified_lead.status.value)
print("crm.leads.convert disponible ?", hasattr(crm.leads, "convert"))

## 8. Définir un Pipeline commercial


In [ ]:
from decimal import Decimal
from pycrmkit.pipelines import Stage, StageOutcome, StageTransition

stages = (
    Stage("new", "New", 0, Decimal("0.10")),
    Stage("qualified", "Qualified", 1, Decimal("0.30")),
    Stage("proposal", "Proposal", 2, Decimal("0.60")),
    Stage("won", "Won", 3, terminal=True, outcome=StageOutcome.WON),
    Stage("lost", "Lost", 4, terminal=True, outcome=StageOutcome.LOST),
)

transitions = (
    StageTransition("new", "qualified"),
    StageTransition("qualified", "proposal"),
    StageTransition("proposal", "won"),
    StageTransition("proposal", "lost"),
)

pipeline = crm.pipelines.define(
    id="sales",
    name="Default Sales Pipeline",
    stages=stages,
    transitions=transitions,
)

print("Pipeline:", pipeline.id, "-", pipeline.name)
for stage in pipeline.stages:
    print(stage.position, stage.id, stage.default_probability, stage.terminal, stage.outcome)


## 9. Convertir le Lead puis faire progresser l'Opportunity

In [ ]:
opportunity = crm.leads.convert(
    qualified_lead.id,
    name="Analytical Engines — CRM rollout",
    estimated_value=Decimal("25000.00"),
    currency="EUR",
    pipeline_id=pipeline.id,
    idempotency_key="notebook-lead-conversion",
)

retry = crm.leads.convert(
    qualified_lead.id,
    name="Analytical Engines — CRM rollout",
    estimated_value=Decimal("25000.0"),
    currency="eur",
    pipeline_id="SALES",
    idempotency_key="notebook-lead-conversion",
)

print("Converted:", opportunity.id, opportunity.stage_id, opportunity.status.value)
print("Idempotent retry returned same Opportunity:", retry.id == opportunity.id)

for target_stage in ("qualified", "proposal", "won"):
    opportunity = crm.opportunities.move(opportunity.id, to=target_stage)
    print(
        f"Moved to {opportunity.stage_id:10} | "
        f"probability={opportunity.probability} | status={opportunity.status.value}"
    )

## 10. Vérifier qu'une transition interdite est rejetée

Ce test crée une seconde Opportunity au stage `new` puis tente `new → won`, transition non déclarée dans le Pipeline.


In [ ]:
from pycrmkit.pipelines import InvalidStageTransition

invalid_demo = crm.opportunities.create(
    name="Invalid transition demo",
    contact_id=contact.id,
    pipeline_id=pipeline.id,
    stage_id="new",
)

try:
    crm.opportunities.move(invalid_demo.id, to="won")
except InvalidStageTransition as exc:
    print("✅ Transition rejetée comme prévu")
    print("Type   :", type(exc).__name__)
    print("Message:", exc)
else:
    raise AssertionError("La transition new → won aurait dû être rejetée.")


## 11. Requêtes simples et pagination


In [ ]:
contacts_page = crm.contacts.search()
pipelines_page = crm.pipelines.list()

print("Contacts :", contacts_page.total)
print("Pipelines:", pipelines_page.total)
print("Has next :", contacts_page.has_next)


## 12. Smoke tests du notebook

Ces assertions donnent un signal rapide que les principaux contrats utilisés ici fonctionnent ensemble.


In [ ]:
assert __version__ == "0.3.0b2"
assert contact.emails[0].normalized == "ada.lovelace@example.com"
assert relationship.role == "CTO"
assert completed_task.status.value == "completed"
assert hasattr(crm.leads, "convert")
assert retry.id == opportunity.id
assert pipeline.initial_stage.id == "new"
assert opportunity.stage_id == "won"
assert opportunity.status.value == "won"
assert opportunity.probability == Decimal("1")

print("✅ PyCRMKit 0.3.0b2 — smoke tests passed")

## 13. Pour aller plus loin localement

Après ce notebook, les vérifications de qualité du dépôt peuvent être lancées depuis la racine :

```bash
pytest
make test
make lint
make typecheck
make coverage
make build
python scripts/release_check.py
```

Le prochain jalon attendu est **`0.3.0rc1 — Sales Integration`**, consacré à la qualification end-to-end et au gel candidat de la surface Sales avant `0.3.0` stable.